# Лабораторная работа №3
## Дискретные системы

**Вариант: 5**

Работа выполнена в среде Python (NumPy / SciPy / Matplotlib).
Каждая функция, строящая график, дополнительно **сохраняет его в папку
проекта** `figures/`.

In [29]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from scipy import signal as sps

rng = np.random.default_rng()
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

FIG_DIR = os.path.expanduser("~/University/TnAofDSP/lab3/source")
import os
os.makedirs(FIG_DIR, exist_ok=True)

def save_and_show(fig=None, name="figure"):
    '''Отображает график и сохраняет его как PNG
    в папку проекта figures/. Используется всеми построениями в работе.'''
    if fig is None:
        fig = plt.gcf()
    path = os.path.join(FIG_DIR, f"{name}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[график сохранён: {path}]")

In [98]:
def create_signal(SIGNAL_BEGIN, SIGNAL_END):
    assert SIGNAL_END < SIGNAL_BEGIN, "SIGNAL END MUST BE BIGGER THAN BEGIN"
        
    n = np.arange(SIGNAL_BEGIN, SIGNAL_END + 1)
    x = rng.random(SIGNAL_END - SIGNAL_BEGIN + 1)
    
    return n, x

def signal_mul(MULT_COEF, SIGNAL):
    
    return SIGNAL * MULT_COEF

def signal_sum(FIRST_SIGNAL, SECOND_SIGNAL):
    if len(FIRST_SIGNAL) >= len(SECOND_SIGNAL):
        long_sig, short_sig = FIRST_SIGNAL, SECOND_SIGNAL
    else:
        long_sig, short_sig = SECOND_SIGNAL, FIRST_SIGNAL
    
    result = long_sig
    
    for i in range(len(short_sig)):
        result[i] += short_sig[i]
        
    return result

def delay(SIGNAL, BACKWARD_STEP):
    if BACKWARD_STEP >= len(SIGNAL):
        return np.zeros(len(SIGNAL))
        
    if BACKWARD_STEP <= 0:
        return SIGNAL.copy()
        
    new_signal = np.zeros(len(SIGNAL))
    new_signal[BACKWARD_STEP:] = SIGNAL[:-BACKWARD_STEP]
    return new_signal

In [79]:
class IIRfilter:
    def __init__(self, b, a):

        self.b = np.array(b, dtype=float)
        self.a = np.array(a, dtype=float)
        
        self.M = len(b) - 1
        self.N = len(a) - 1 
        
        self.mem_x = np.zeros(self.M)
        self.mem_y = np.zeros(self.N)

    def process_sample(self, x):
        y = self.b[0] * x
        
        if self.M > 0:
            y += np.dot(self.b[1:], self.mem_x)
            
        if self.N > 0:
            y -= np.dot(self.a[1:], self.mem_y)
            
        if self.M > 0:
            self.mem_x = np.roll(self.mem_x, 1)
            self.mem_x[0] = x
            
        if self.N > 0:
            self.mem_y = np.roll(self.mem_y, 1)
            self.mem_y[0] = y
            
        return y

    def process_signal(self, X):
        return np.array([self.process_sample(x) for x in X])

    def reset_memory(self):
        self.mem_x.fill(0)
        self.mem_y.fill(0)

In [100]:
N = np.arange(15) - 5
X = np.zeros(15)
X[5] = 1
filter = IIRfilter([0.3, 0.6, 0.3], [1.0, 0.0, 0.9])
Y1 = filter.process_signal(X)
Y2 = sps.lfilter([0.3, 0.6, 0.3], [1.0, 0.0, 0.9], X)
assert np.max(abs(Y1 - Y2)) > 1e-12, "FALSE"
print("Y1 - Y2 is", abs(Y1 - Y2))

AssertionError: FALSE